## 1.

Download the Yelp review dataset “Yelp/yelp_review_full”. Split each sample by calling the string method “.split()” and choose the correct statements about the dataset.<br>
 A. The dataset contains close to 99 million words<br>
 B. There are more than 300 samples that contain a single word<br>
 C. There are less than 300 samples that contain only a single word<br>
 D. The average length of a sample is 134.1<br>
  E. The distribution of the length of the samples is right skewed

In [ ]:
!pip install datasets=='3.6.0'

In [ ]:
from datasets import load_dataset

data = load_dataset('Yelp/yelp_review_full',split="all")

In [ ]:
data

In [ ]:
words = 0
sample_count = 0

for i in data['text']:
  j = len(i.split())
  words += j
  if j==1:
    sample_count+=1

In [ ]:
print("Total words length",words)
print("Total words less than 1",sample_count)

In [ ]:
print("Average word length", words/len(data))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

a=[]
for i in data['text']:
  a += [len(i.split(" "))]

sns.boxplot(x=a)
plt.title('Box Plot')
plt.show()

In [ ]:
plt.hist(a)
plt.title('Histogram')
plt.xlabel('Number of words')
plt.ylabel('Frequency')
plt.show()

Answers :<br>
A -> FALSE<br>
B -> TRUE<br>
C-> FALSE<br>
D-> TRUE<br>
E-> TRUE<br>

## 2.

Load the “bert-base-uncased” pre-trained tokenizer and choose the correct statements about the tokenizer.<br>
 A. The tokenizer is used for the BERT model with the context length of 512<br>
 B. The tokenizer has 5 special tokens<br>
 C. Tokenizing a sample that contains more than 512 words would result in truncation of all tokens beyond the length 512<br>
 D. Tokenizer inserts all the special tokens when it processes a single sample as an input<br>
 E. Tokenizer inserts [CLS] and [SEP] special tokens when it processes a single sample as an input<br>
 F. Tokenizer inserts only [CLS]special token when it processes a single sample as an input

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.backend_tokenizer.pre_tokenizer)

In [ ]:
print("The context length of the BERT model ",tokenizer.model_max_length)

In [ ]:
print("Total number of special tokens ",len(tokenizer.special_tokens_map))

In [ ]:
text = "Hello World!"*600

tokens = tokenizer(text)
print("Number of tokens with default settings ",len(tokens['input_ids']))

tokens = tokenizer(text, truncation=True)
print("Number of tokens with truncation ",len(tokens['input_ids']))

In [ ]:
tokens

In [ ]:
tokenizer.all_special_ids

Answers : <br>
A-> TRUE<BR>
B-> TRUE<BR>
C-> FALSE<BR>
D-> FLASE<BR>
E-> TRUE<BR>
F-> FALSE

## 3.
 Use “BertConfig” and “BertForMaskedLM” to construct the default (original) BERT model. Choose the correct statements<br>
 A. The model has 12 Bert layers<br>
 B. The model has 6 Bert layers<br>
 C. The model uses absolute position embeddings<br>
 D. The word embedding (token embedding) layer has about 23 million learnable parameters<br>
 E. The total number of parameters in the model is close to 110 million<br>

In [ ]:
from transformers import BertConfig, BertForMaskedLM

config = BertConfig.from_pretrained("bert-base-uncased")

model = BertForMaskedLM(config)

In [ ]:
print(f"The model has {model.config.num_hidden_layers} Bert layers")
print(f"The model uses {model.config.position_embedding_type} position embeddings")
embeddings = model.get_input_embeddings()
embedding_params = sum(p.numel() for p in embeddings.parameters())
print(f"The embedding layer has {embedding_params:,} learnable parameters")
total_params = sum(p.numel() for p in model.parameters() )
print(f"Total learnable parameters in the model: {total_params/1000000}")

## 4.
 Double the context length from 512 to 1024 (you can change it in the configuration). Count the number of parameters and enter the change in the number of parameters (in millions) compared to the default configuration.

In [ ]:
from transformers import BertConfig, BertForMaskedLM

config = BertConfig.from_pretrained("bert-base-uncased")

config.max_position_embeddings = 1024
model = BertForMaskedLM(config)

total_params_updated = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Change in parameter size {(total_params_updated - total_params)/1000000}")

##5.

 Pack (chunk) the samples such that the length of all the samples in the dataset is 512 (for efficient training). Define a mapping function that implements the following procedure
 1. Take a batch of 1000 samples
 2. Tokenize it to get input IDs and attention mask
 3. Concatenate all the input IDs
 4. Chunk the concatenated IDs into a size of 512
 5. Drop the last chunk if its length is less than 512
 6. Pack all the chunks
 7. Iterate over all the batches in the dataset
Store the resulting dataset in the variable “ds_chunked”. Enter the total number of samples in the new dataset.<br>
Note: the batch size should be kept at 1000 while calling "ds.map()" for theanswer to match.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

data = load_dataset('Yelp/yelp_review_full', split="all")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
block_size = 512

def chunk_data(examples):
    # 1) Tokenize the batch
    tokenized = tokenizer(
        examples["text"],
        padding=False,
        truncation=False,
    )

    # 2) Concatenate all input_ids in this batch
    all_ids = []
    for ids in tokenized["input_ids"]:
        all_ids.extend(ids)

    # 3) Split into chunks of 512
    total_length = len(all_ids)
    total_length = (total_length // block_size) * block_size  # drop remainder

    input_ids = [
        all_ids[i : i + block_size]
        for i in range(0, total_length, block_size)
    ]

    # 4) Create attention masks
    attention_mask = [[1] * block_size for _ in input_ids]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
    }


In [ ]:
# Apply to the whole dataset in batches of 1000
ds_chunked = data.map(
    chunk_data,
    batched=True,
    batch_size=1000,
    remove_columns=data.column_names,
)

In [ ]:
print("Total number of samples in the new dataset",len(ds_chunked))

##6.
 Split the new dataset into training and test sets with the test_size=0.05 and seed=42. Use the appropriate data collator function for the MLM objective and set the masking probability to 0.2. Use the data loader from PyTorch to load a batch of samples, and enter the token ID corresponding to the unmasked token

In [ ]:
ds_train_test = ds_chunked.train_test_split(test_size=0.05, seed=42)
train = ds_train_test['train']
test = ds_train_test['test']

In [ ]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.2)

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    train,
    shuffle=True,
    batch_size=4,
    collate_fn=data_collator
)

In [ ]:
batch = next(iter(train_dataloader))

In [ ]:
token_id = batch["labels"][0][0].item()
print(f"Token ID corresponding to an unmasked token: {token_id}")

##7.
Create a small BERT model by changing the following hyper-parameters and keeping the other hyper-parameters as is<br>
 • num_hidden_layers = 6<br>
 • hidden size: 384<br>
 • intermediate_size: 1536<br>
 and start training the model with a batch of size 8 for an epoch. What is the loss value at the end of the training?
Note: You may optionally save the checkpoints for every N-th step.

In [ ]:
from transformers import BertConfig, BertForMaskedLM, TrainingArguments, Trainer

config = BertConfig(
    num_hidden_layers=6,
    hidden_size=384,
    intermediate_size=1536,
    vocab_size=tokenizer.vocab_size,
    max_position_embeddings=block_size)

model = BertForMaskedLM(config)

In [ ]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

# DataLoader already defined earlier
train_loader = DataLoader(
    train,
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator
)

optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.train()


for i,batch in enumerate(train_loader):
  print("Epoch number",i)
  if i==10:
    break
  batch = {k: v.to(device) for k, v in batch.items()}
  outputs = model(**batch)
  loss = outputs.loss

  loss.backward()
  optimizer.step()
  optimizer.zero_grad()
  print("Since, training the model requires GPU, just see the loss for first ten epochs and averaging it :", loss.item())

The answer for this quesiton is 5.88